# Polynomial Regression: Optimization Algorithms Comparison

**Course Project | NumPy from Scratch**

This notebook compares six optimization algorithms for Polynomial Regression across five real Kaggle datasets.

### Research question
How do Gradient Descent, Stochastic Gradient Descent, Mini-Batch Gradient Descent, RMSprop, Adam, and Momentum compare in convergence and predictive performance on different real-world regression datasets?

### Important implementation note
The polynomial regression model, loss function, gradients, and all six optimizers are implemented with **NumPy**. Pandas is used only for data loading/preparation and Matplotlib for visualization. No Scikit-learn, TensorFlow, or PyTorch model/optimizer is used.

## 1. Datasets and Kaggle input setup

Attach these five datasets to this Kaggle Notebook using **Add Input**:

| Dataset | Kaggle source | Expected CSV | Feature (X) | Target (y) |
|---|---|---|---|---|
| Salary | [Salary Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/Salary_dataset) | `Salary_dataset.csv` | `YearsExperience` | `Salary` |
| Auto MPG | [Auto MPG](https://www.kaggle.com/datasets/uciml/autompg-dataset) | `auto-mpg.csv` | `horsepower` | `mpg` |
| California Housing | [California Housing Prices](https://www.kaggle.com/datasets/camnugent/california-housing-prices) | `housing.csv` | `median_income` | `median_house_value` |
| Medical Insurance | [Medical Cost Personal Datasets](https://www.kaggle.com/datasets/mirichoi0218/insurance) | `insurance.csv` | `age` | `charges` |
| Bike Sharing | [Bike Sharing Demand](https://www.kaggle.com/competitions/bike-sharing-demand/data) | `train.csv` | `temp` | `count` |

Bike Sharing is a Kaggle competition dataset and may require accepting its competition rules before adding the input.

We use one numerical feature per dataset to keep the polynomial relationship interpretable and comparable. This is a univariate polynomial regression experiment, not a full-feature prediction benchmark.

In [ ]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Locate and load the Kaggle CSV files

The loader searches the Kaggle input directory recursively. If your downloaded file has a different name, update the `filename` field in the configuration below.

In [ ]:
def find_kaggle_file(filename):
    search_roots = ["/kaggle/input", "."]
    matches = []
    for root in search_roots:
        if os.path.exists(root):
            matches.extend(glob.glob(os.path.join(root, "**", filename), recursive=True))
    matches = [p for p in matches if os.path.isfile(p)]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Add the corresponding Kaggle dataset via "
            "Add Input, then check the expected filename."
        )
    return matches[0]

DATASETS = {
    "Salary": {
        "filename": "Salary_dataset.csv",
        "x": "YearsExperience",
        "y": "Salary",
        "x_aliases": ["YearsExperience", "Years Experience", "experience"],
        "y_aliases": ["Salary", "salary"]
    },
    "Auto MPG": {
        "filename": "auto-mpg.csv",
        "x": "horsepower",
        "y": "mpg",
        "x_aliases": ["horsepower", "Horsepower"],
        "y_aliases": ["mpg", "MPG"]
    },
    "California Housing": {
        "filename": "housing.csv",
        "x": "median_income",
        "y": "median_house_value",
        "x_aliases": ["median_income", "MedInc"],
        "y_aliases": ["median_house_value", "MedHouseVal"]
    },
    "Medical Insurance": {
        "filename": "insurance.csv",
        "x": "age",
        "y": "charges",
        "x_aliases": ["age", "Age"],
        "y_aliases": ["charges", "Charges"]
    },
    "Bike Sharing": {
        "filename": "train.csv",
        "x": "temp",
        "y": "count",
        "x_aliases": ["temp", "temperature"],
        "y_aliases": ["count", "cnt"]
    }
}

def select_column(df, aliases, requested):
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for name in aliases + [requested]:
        key = str(name).strip().lower()
        if key in normalized:
            return normalized[key]
    raise KeyError(f"Could not find any of {aliases + [requested]}. Available columns: {list(df.columns)}")

def load_dataset(name, config):
    path = find_kaggle_file(config["filename"])
    df = pd.read_csv(path)
    x_col = select_column(df, config["x_aliases"], config["x"])
    y_col = select_column(df, config["y_aliases"], config["y"])

    # Convert to numeric; values such as '?' become NaN.
    clean = df[[x_col, y_col]].copy()
    clean[x_col] = pd.to_numeric(clean[x_col], errors="coerce")
    clean[y_col] = pd.to_numeric(clean[y_col], errors="coerce")
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna()

    # Keep the experiment computationally manageable while preserving reproducibility.
    if len(clean) > 12000:
        clean = clean.sample(n=12000, random_state=42)

    X = clean[x_col].to_numpy(dtype=float).reshape(-1, 1)
    y = clean[y_col].to_numpy(dtype=float).reshape(-1, 1)
    return X, y, clean, path, x_col, y_col

loaded_data = {}
for name, cfg in DATASETS.items():
    try:
        X, y, df, path, x_col, y_col = load_dataset(name, cfg)
        loaded_data[name] = {
            "X": X, "y": y, "df": df, "path": path,
            "x_col": x_col, "y_col": y_col
        }
        print(f"{name:22s} | rows={len(df):5d} | X={x_col:22s} | y={y_col:22s} | {path}")
    except Exception as e:
        print(f"ERROR loading {name}: {e}")

if len(loaded_data) != 5:
    raise RuntimeError(
        f"Only {len(loaded_data)} of 5 datasets loaded. Review the Kaggle inputs and filenames."
    )

## 3. Exploratory data analysis

We inspect sample size, summary statistics, missing values after numeric conversion, and the raw feature-target relationship.

In [ ]:
summary_rows = []
for name, item in loaded_data.items():
    X, y = item["X"], item["y"]
    summary_rows.append({
        "Dataset": name,
        "Samples": len(X),
        "X minimum": float(np.min(X)),
        "X maximum": float(np.max(X)),
        "y minimum": float(np.min(y)),
        "y maximum": float(np.max(y))
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
for ax, (name, item) in zip(axes, loaded_data.items()):
    ax.scatter(item["X"], item["y"], alpha=0.45, s=14)
    ax.set_title(name)
    ax.set_xlabel(item["x_col"])
    ax.set_ylabel(item["y_col"])
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 4. Train/test split and scaling (NumPy)

Each dataset is shuffled using the same fixed random seed and split into 80% training and 20% testing data. Feature scaling and target scaling are fitted on the training set only to prevent data leakage.

Polynomial features are generated as:

\[
\phi(x) = [1, z, z^2, \ldots, z^d]
\]

where \(z\) is the standardized input feature and \(d=3\) by default.

In [ ]:
DEGREE = 3
TEST_SIZE = 0.20
RANDOM_STATE = 42

def train_test_split_numpy(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    n_test = max(1, int(round(len(X) * test_size)))
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def fit_standardizer(a):
    mean = np.mean(a, axis=0)
    std = np.std(a, axis=0)
    std[std == 0] = 1.0
    return mean, std

def transform_standardizer(a, mean, std):
    return (a - mean) / std

def polynomial_features(X_scaled, degree=3):
    n = X_scaled.shape[0]
    columns = [np.ones((n, 1))]
    for power in range(1, degree + 1):
        columns.append(X_scaled ** power)
    return np.hstack(columns)

prepared = {}
for name, item in loaded_data.items():
    X_train, X_test, y_train, y_test = train_test_split_numpy(
        item["X"], item["y"], TEST_SIZE, RANDOM_STATE
    )
    x_mean, x_std = fit_standardizer(X_train)
    y_mean, y_std = fit_standardizer(y_train)

    X_train_scaled = transform_standardizer(X_train, x_mean, x_std)
    X_test_scaled = transform_standardizer(X_test, x_mean, x_std)
    y_train_scaled = transform_standardizer(y_train, y_mean, y_std)
    y_test_scaled = transform_standardizer(y_test, y_mean, y_std)

    prepared[name] = {
        "X_train_raw": X_train, "X_test_raw": X_test,
        "y_train_raw": y_train, "y_test_raw": y_test,
        "X_train_poly": polynomial_features(X_train_scaled, DEGREE),
        "X_test_poly": polynomial_features(X_test_scaled, DEGREE),
        "y_train_scaled": y_train_scaled, "y_test_scaled": y_test_scaled,
        "x_mean": x_mean, "x_std": x_std, "y_mean": y_mean, "y_std": y_std
    }

print(f"Prepared {len(prepared)} datasets with polynomial degree {DEGREE}.")

## 5. Polynomial regression model and loss function

The model is:

\[
\hat{y} = X_{poly}\theta
\]

We minimize Mean Squared Error:

\[
J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})^2
\]

The gradient is:

\[
\nabla_\theta J(\theta) = \frac{1}{m}X_{poly}^{T}(X_{poly}\theta-y)
\]

The factor \(1/2\) in the loss simplifies the derivative.

In [ ]:
def predict(X_poly, theta):
    return X_poly @ theta

def compute_loss(X_poly, y, theta):
    errors = predict(X_poly, theta) - y
    return float(np.mean(errors ** 2) / 2.0)

def compute_gradient(X_batch, y_batch, theta):
    m = len(y_batch)
    errors = X_batch @ theta - y_batch
    return (X_batch.T @ errors) / m

## 6. Implement six optimizers from scratch using NumPy

All optimizers use the same model, polynomial degree, train/test split, initialization, and standardized data. Loss history is recorded once per epoch.

- **GD:** uses the full training set for each update.
- **SGD:** updates using one randomly selected training example at a time.
- **Mini-Batch GD:** updates using small batches.
- **RMSprop:** scales the learning rate using an exponential moving average of squared gradients.
- **Adam:** combines moving averages of gradients and squared gradients with bias correction.
- **Momentum:** accumulates a fraction of the previous update to smooth and accelerate descent.

In [ ]:
def optimize(X, y, method, epochs=500, learning_rate=0.01,
             batch_size=32, beta=0.9, beta2=0.999, epsilon=1e-8):
    n_samples, n_features = X.shape
    theta = np.zeros((n_features, 1), dtype=float)
    history = []

    velocity = np.zeros_like(theta)       # Momentum / Adam first moment
    squared_avg = np.zeros_like(theta)    # RMSprop / Adam second moment
    timestep = 0

    rng = np.random.default_rng(42)

    for epoch in range(epochs):
        if method == "GD":
            grad = compute_gradient(X, y, theta)
            theta -= learning_rate * grad

        elif method == "SGD":
            order = rng.permutation(n_samples)
            for idx in order:
                X_batch = X[idx:idx+1]
                y_batch = y[idx:idx+1]
                grad = compute_gradient(X_batch, y_batch, theta)
                theta -= learning_rate * grad

        elif method in ["Mini-Batch GD", "Momentum", "RMSprop", "Adam"]:
            order = rng.permutation(n_samples)
            for start in range(0, n_samples, batch_size):
                batch_idx = order[start:start + batch_size]
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]
                grad = compute_gradient(X_batch, y_batch, theta)
                timestep += 1

                if method == "Mini-Batch GD":
                    theta -= learning_rate * grad

                elif method == "Momentum":
                    velocity = beta * velocity + learning_rate * grad
                    theta -= velocity

                elif method == "RMSprop":
                    squared_avg = beta * squared_avg + (1 - beta) * (grad ** 2)
                    theta -= learning_rate * grad / (np.sqrt(squared_avg) + epsilon)

                elif method == "Adam":
                    velocity = beta * velocity + (1 - beta) * grad
                    squared_avg = beta2 * squared_avg + (1 - beta2) * (grad ** 2)
                    m_hat = velocity / (1 - beta ** timestep)
                    v_hat = squared_avg / (1 - beta2 ** timestep)
                    theta -= learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)

        else:
            raise ValueError(f"Unknown optimizer: {method}")

        history.append(compute_loss(X, y, theta))

    return theta, np.asarray(history)

OPTIMIZERS = ["GD", "SGD", "Mini-Batch GD", "RMSprop", "Adam", "Momentum"]

## 7. Evaluation metrics

We report test-set MSE, RMSE, MAE, and R² in the original target units. The metrics are calculated with NumPy.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

## 8. Run all 30 experiments

There are 5 datasets × 6 optimizers = 30 experiments. We record training time and test metrics for each run.

In [ ]:
EPOCHS = 500
LEARNING_RATE = 0.01
BATCH_SIZE = 32

all_results = []
trained_models = {}
loss_histories = {}

for dataset_name, data in prepared.items():
    print(f"Running: {dataset_name}")
    for optimizer_name in OPTIMIZERS:
        start_time = time.perf_counter()
        theta, history = optimize(
            data["X_train_poly"],
            data["y_train_scaled"],
            method=optimizer_name,
            epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            batch_size=BATCH_SIZE
        )
        elapsed = time.perf_counter() - start_time

        y_pred_scaled = predict(data["X_test_poly"], theta)
        y_pred = y_pred_scaled * data["y_std"] + data["y_mean"]
        metrics = regression_metrics(data["y_test_raw"], y_pred)

        all_results.append({
            "Dataset": dataset_name,
            "Optimizer": optimizer_name,
            **metrics,
            "Training Time (s)": elapsed,
            "Final Training Loss (scaled)": history[-1]
        })
        trained_models[(dataset_name, optimizer_name)] = theta
        loss_histories[(dataset_name, optimizer_name)] = history

results_df = pd.DataFrame(all_results)
display(results_df.sort_values(["Dataset", "RMSE"]).round(4))
print("Completed experiments:", len(results_df))

## 9. Compare convergence (training loss)

The curves show how the training loss changes across epochs. Note that SGD and mini-batch methods make more frequent parameter updates within an epoch than full-batch GD, so epoch-level curves are a practical comparison rather than an exact update-count comparison.

In [ ]:
for dataset_name in prepared:
    plt.figure(figsize=(11, 6))
    for optimizer_name in OPTIMIZERS:
        history = loss_histories[(dataset_name, optimizer_name)]
        plt.plot(history, label=optimizer_name, linewidth=1.8)
    plt.title(f"Training Loss Comparison — {dataset_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Training MSE Loss (standardized target)")
    plt.yscale("log")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 10. Actual vs. predicted values

For each dataset, we visualize the test observations and each optimizer's predictions. The predictions are converted back to the original target units.

In [ ]:
for dataset_name, data in prepared.items():
    plt.figure(figsize=(10, 6))
    x_test = data["X_test_raw"].reshape(-1)
    y_test = data["y_test_raw"].reshape(-1)
    order = np.argsort(x_test)

    plt.scatter(x_test, y_test, alpha=0.45, s=18, label="Actual test data")

    for optimizer_name in OPTIMIZERS:
        theta = trained_models[(dataset_name, optimizer_name)]
        X_scaled = transform_standardizer(data["X_test_raw"], data["x_mean"], data["x_std"])
        X_poly = polynomial_features(X_scaled, DEGREE)
        pred_scaled = predict(X_poly, theta)
        pred = (pred_scaled * data["y_std"] + data["y_mean"]).reshape(-1)
        plt.plot(x_test[order], pred[order], linewidth=1.5, label=optimizer_name)

    plt.title(f"Actual vs Predicted — {dataset_name}")
    plt.xlabel(data.get("x_col", "Input feature"))
    plt.ylabel("Target")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 11. Overall optimizer comparison

Because target units and scales differ between datasets, raw RMSE values should be compared **within each dataset**, not directly across different datasets. We also calculate each optimizer's average rank across datasets using RMSE, where rank 1 is best.

In [ ]:
ranked = results_df.copy()
ranked["RMSE Rank"] = ranked.groupby("Dataset")["RMSE"].rank(method="min", ascending=True)
optimizer_summary = (
    ranked.groupby("Optimizer")
    .agg(
        Average_RMSE_Rank=("RMSE Rank", "mean"),
        Average_Training_Time_s=("Training Time (s)", "mean"),
        Average_R2=("R2", "mean")
    )
    .sort_values("Average_RMSE_Rank")
)
display(optimizer_summary.round(4))

pivot_rmse = results_df.pivot(index="Dataset", columns="Optimizer", values="RMSE")
display(pivot_rmse.round(3))

plt.figure(figsize=(12, 6))
optimizer_summary["Average_RMSE_Rank"].sort_values().plot(kind="bar")
plt.title("Average RMSE Rank Across Five Datasets (lower is better)")
plt.ylabel("Average rank")
plt.xlabel("Optimizer")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 12. Best optimizer per dataset

The following table identifies the optimizer with the lowest test RMSE for each dataset. The conclusion is generated from the experiment results rather than assuming that one optimizer must always win.

In [ ]:
best_per_dataset = (
    results_df.loc[results_df.groupby("Dataset")["RMSE"].idxmin(),
                   ["Dataset", "Optimizer", "RMSE", "MAE", "R2", "Training Time (s)"]]
    .sort_values("Dataset")
    .reset_index(drop=True)
)
display(best_per_dataset.round(4))

overall_best = optimizer_summary.index[0]
print(f"Best average RMSE rank across these experiments: {overall_best}")
print("Interpret results per dataset: the same optimizer is not guaranteed to win every dataset.")

## 13. Discussion and limitations

1. Polynomial regression can model curved relationships, but a single input feature cannot capture all factors affecting real-world targets.
2. A degree-3 polynomial may underfit or overfit depending on the dataset. Degree selection should be validated separately if the assignment requires hyperparameter tuning.
3. Learning rates affect convergence. This experiment uses a common learning rate for a straightforward comparison; it is not an exhaustive hyperparameter search.
4. The datasets differ in size, noise, and relationship shape. Compare optimizers within each dataset and use average ranks for a broad summary.
5. For large datasets, the code caps the sample at 12,000 rows for runtime consistency. Remove or change this cap in `load_dataset` if full-data training is required.
6. Bike Sharing's `count` target is strongly influenced by time, weather, and calendar features. Using temperature alone is intentionally a univariate polynomial-regression exercise, not a production-quality demand forecast.
7. The California Housing and insurance datasets may contain influential observations; polynomial models can be sensitive to outliers.

### Final conclusion
The experiment evaluates six NumPy-based optimization algorithms over five real Kaggle datasets, producing 30 model runs. The best optimizer is selected empirically using test RMSE and average RMSE rank. Report the observed results from the executed notebook rather than claiming a predetermined winner.

## 14. Export results

This cell saves the experiment tables to the notebook's working directory. In Kaggle, download these files from the Output panel after running the notebook.

In [ ]:
results_df.to_csv("optimizer_results_all_datasets.csv", index=False)
optimizer_summary.to_csv("optimizer_summary.csv")
best_per_dataset.to_csv("best_optimizer_per_dataset.csv", index=False)

print("Saved:")
print("- optimizer_results_all_datasets.csv")
print("- optimizer_summary.csv")
print("- best_optimizer_per_dataset.csv")

## 15. References

- Kaggle Salary Dataset: https://www.kaggle.com/datasets/iamsouravbanerjee/Salary_dataset
- Kaggle Auto MPG Dataset: https://www.kaggle.com/datasets/uciml/autompg-dataset
- Kaggle California Housing Prices: https://www.kaggle.com/datasets/camnugent/california-housing-prices
- Kaggle Medical Cost Personal Datasets: https://www.kaggle.com/datasets/mirichoi0218/insurance
- Kaggle Bike Sharing Demand: https://www.kaggle.com/competitions/bike-sharing-demand/data

**Reproducibility:** random seed = 42. The notebook uses Python, NumPy, Pandas, and Matplotlib.